# IndexCalc — Phase 6e 테스트

Functional Components + JAX Autodiff: 텐서를 좌표 함수로 제공하면 `∂_μ V^ν`를 `jax.jacfwd`로 자동 계산.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import jax
import jax.numpy as jnp
from indexcalc import (
    IndexSpace, Tensor, IndexRegistry, to_latex, evaluate,
    partial, expand_partial,
    LeviCivitaConnection, covariant, expand_covariant,
)
from IPython.display import display, Math

In [ ]:
spacetime = IndexSpace("spacetime", dim=4, indices="μνλρσ", metric="g")

# 좌표점
x0 = jnp.array([2., jnp.pi/6, 0., 0.])

# 함수형 component
def V_func(x):
    """V^μ = (x⁰², sin(x¹), 0, 0)"""
    return jnp.array([x[0]**2, jnp.sin(x[1]), 0., 0.])

## 1. 함수형 component: $V^\mu(x)$

배열 대신 함수를 넘기면 `coords`에서 자동 평가.

In [ ]:
V_t = Tensor("V", [spacetime.upper("μ")])

result1 = evaluate(V_t, {"V": V_func}, backend="jax", coords=x0)
print(f"x = {x0}")
print(f"V(x) = {result1}")
print(f"기대: [4, 0.5, 0, 0]  (2²=4, sin(π/6)=0.5)")

## 2. JAX Autodiff: $\partial_\nu V^\mu$

`"∂V"` 키가 없으면 `jax.jacfwd`로 자동 미분 — 핵심 기능!

In [ ]:
dV = partial(V_t, spacetime.lower("ν"))
display(Math(to_latex(dV)))

result2 = evaluate(dV, {"V": V_func}, backend="jax", coords=x0)
print(f"∂_ν V^μ (autodiff) =\n{result2}")
print(f"\n∂_0 V^0 = 2x⁰ = {2*float(x0[0]):.3f}  →  결과: {float(result2[0,0]):.3f}")
print(f"∂_1 V^1 = cos(x¹) = {float(jnp.cos(x0[1])):.3f}  →  결과: {float(result2[1,1]):.3f}")

## 3. 공변미분 + autodiff: $\nabla_\mu V^\nu$

`expand_covariant()` 후 evaluate — `∂V`는 autodiff, `Γ`는 배열로 혼합 사용.

In [ ]:
g     = Tensor("g", [spacetime.lower("μ"), spacetime.lower("ν")])
g_inv = Tensor("g", [spacetime.upper("μ"), spacetime.upper("ν")])
christoffel = LeviCivitaConnection(g, g_inv, spacetime)

V_up = Tensor("V", [spacetime.upper("ν")])
mu = spacetime.lower("μ")
nabla_V = covariant(V_up, mu, christoffel)
expanded = expand_covariant(nabla_V)

display(Math(r"\nabla_\mu V^\nu = " + to_latex(expanded)))

# Nonzero Γ: Γ^0_{10} = 0.5
Gamma = jnp.zeros((4, 4, 4)).at[0, 1, 0].set(0.5)

result3 = evaluate(expanded, {"V": V_func, "Γ": Gamma},
                   backend="jax", coords=x0)
print(f"∇_μ V^ν =\n{result3}")
print(f"\n(1,0) 성분 = {float(result3[1,0]):.3f}")
print(f"= ∂_1 V^0 + Γ^0_{{10}} V^0 = 0 + 0.5 × {float(x0[0]**2)} = {0.5*float(x0[0]**2):.3f}")

## 4. 2차 미분: $\partial_\mu \partial_\nu f$

스칼라 함수의 Hessian을 자동 계산.

In [ ]:
def f_func(x):
    """f = x⁰ x¹ + (x¹)³"""
    return x[0] * x[1] + x[1]**3

f_T = Tensor("f", [])  # scalar
df = partial(f_T, spacetime.lower("μ"))
ddf = partial(df, spacetime.lower("ν"))

display(Math(to_latex(ddf)))

result4 = evaluate(ddf, {"f": f_func}, backend="jax", coords=x0)
print(f"∂_μ ∂_ν f =\n{result4}")
print(f"\n∂₀∂₁f = 1 (= ∂/∂x⁰ of x⁰) → {float(result4[0,1]):.1f}")
print(f"∂₁∂₁f = 6x¹ = {6*float(x0[1]):.3f} → {float(result4[1,1]):.3f}")

## 5. Explicit override: `"∂V"` 키가 있으면 autodiff보다 우선

사용자가 `"∂V"`를 직접 제공하면 autodiff 대신 해당 배열을 사용.

In [ ]:
# "∂V" 키가 있으면 → 그걸 사용 (autodiff 안 함)
manual_dV = jnp.eye(4) * 999.  # 일부러 엉뚱한 값

result5_override = evaluate(dV, {"V": V_func, "∂V": manual_dV},
                            backend="jax", coords=x0)
print(f"explicit '∂V' 제공 시:\n{result5_override}")
print("→ 999 값 사용됨 (autodiff가 아닌 explicit override)")

# "∂V" 없으면 → autodiff
result5_auto = evaluate(dV, {"V": V_func}, backend="jax", coords=x0)
print(f"\nautodiff 결과:\n{result5_auto}")
print("→ 실제 미분값 사용됨")